# K-IFRS Agentic RAG Chain

Hybrid Retriever → Reranker → **LLM ↔ Tool 루프**

- **Retriever**: PgVectorRetriever + BM25Retriever → EnsembleRetriever (Top-30)
- **Reranker**: LocalReranker 또는 CohereReranker (Top-3)
- **LLM**: Claude (bind_tools로 4개 tool 제공)
  - `fetch_paragraphs`: 특정 문단 직접 조회 (교차참조 해소)
  - `find_referencing_chunks`: 역방향 검색 (해당 기준서를 참조하는 다른 청크)
  - `explore_related_standards`: 기준서 참조 그래프 탐색
  - `fetch_term_definitions`: 기준서 용어정의(Appendix A) 조회
- **Graph**: retrieve → rerank → generate ⇄ tool_executor → END

In [1]:
import os
import json
import glob
import time
import pickle
import hashlib
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_upstage import UpstageEmbeddings
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

from search.config import CHUNKS_DIR, MODEL_NAME
from search.retriever import (
    PgVectorRetriever, load_child_documents, kiwi_tokenize,
    expand_to_parents, format_parent_context,
)
from search.reranker import get_reranker
from search.query_router import classify_query, apply_authority_boost
from search.tools import TOOL_SCHEMAS, dispatch_tool
from search.db import close_pool

load_dotenv()
print("패키지 로드 완료")

/home/shin/Home/Study/_database/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


패키지 로드 완료


## 1. 설정

In [2]:
RETRIEVAL_K = 30        # Hybrid 검색 후보 수
RERANK_TOP_N = 15       # Reranker 후 parent 그룹핑 후보 수
MAX_CONTEXT_CHARS = 12000  # Parent 확장 시 컨텍스트 총 문자 예산
MAX_PARENTS = 7         # 최대 parent 그룹 수

## 2. Hybrid Retriever 구성

In [3]:
# BM25 캐시 관리
BM25_CACHE_DIR = Path(".")
BM25_PKL_PATH = BM25_CACHE_DIR / "bm25_retriever.pkl"
BM25_META_PATH = BM25_CACHE_DIR / "bm25_cache_meta.json"


def _docs_fingerprint(docs: list[Document]) -> str:
    """문서 수 + 첫/마지막 문서 내용으로 MD5 fingerprint 생성"""
    h = hashlib.md5()
    h.update(str(len(docs)).encode())
    if docs:
        h.update(docs[0].page_content.encode())
        h.update(docs[-1].page_content.encode())
    return h.hexdigest()


def load_or_build_bm25(
    docs: list[Document], preprocess_func, k: int
) -> BM25Retriever:
    fingerprint = _docs_fingerprint(docs)

    # 캐시 유효성 검사
    if BM25_PKL_PATH.exists() and BM25_META_PATH.exists():
        meta = json.loads(BM25_META_PATH.read_text(encoding="utf-8"))
        if meta.get("fingerprint") == fingerprint:
            print(f"  캐시 적중 — {BM25_PKL_PATH} 로드 중...")
            t0 = time.time()
            with open(BM25_PKL_PATH, "rb") as f:
                retriever = pickle.load(f)
            retriever.k = k
            print(f"  BM25 캐시 로드 완료 ({time.time() - t0:.1f}초)")
            return retriever
        else:
            print("  캐시 fingerprint 불일치 — 재빌드합니다.")

    # 캐시 미스: 새로 빌드
    print(f"  BM25 인덱스 빌드 중... (문서 {len(docs)}개)")
    t0 = time.time()
    retriever = BM25Retriever.from_documents(
        docs, preprocess_func=preprocess_func, k=k
    )
    elapsed = time.time() - t0
    print(f"  BM25 인덱스 빌드 완료 ({elapsed:.1f}초)")

    # 캐시 저장
    with open(BM25_PKL_PATH, "wb") as f:
        pickle.dump(retriever, f)
    BM25_META_PATH.write_text(
        json.dumps({"fingerprint": fingerprint, "doc_count": len(docs)},
                   ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"  캐시 저장 완료 — {BM25_PKL_PATH}")
    return retriever


print("load_or_build_bm25 정의 완료")

load_or_build_bm25 정의 완료


In [4]:
# 초기화
print("[1/4] 임베딩 모델 초기화...")
embeddings = UpstageEmbeddings(
    model=MODEL_NAME,
    upstage_api_key=os.getenv("UPSTAGE_API_KEY"),
)

print("[2/4] BM25 인덱스 구축...")
docs = load_child_documents(CHUNKS_DIR)
print(f"  LangChain Document: {len(docs)}개")
bm25_retriever = load_or_build_bm25(docs, preprocess_func=kiwi_tokenize, k=RETRIEVAL_K)

print("[3/4] Hybrid Retriever 구성...")
dense_retriever = PgVectorRetriever(
    embeddings=embeddings, k=RETRIEVAL_K,
)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)

print("[4/4] Reranker 초기화...")
reranker = get_reranker()  # RERANKER_TYPE 환경변수 참조
print(f"  Reranker: {type(reranker).__name__}")
print("초기화 완료")

[1/4] 임베딩 모델 초기화...
[2/4] BM25 인덱스 구축...
  LangChain Document: 16052개
  캐시 적중 — bm25_retriever.pkl 로드 중...
  BM25 캐시 로드 완료 (0.3초)
[3/4] Hybrid Retriever 구성...
[4/4] Reranker 초기화...
  Reranker: CohereReranker
초기화 완료


## 3. LangGraph RAG Chain

In [5]:
# LLM
llm_model = os.getenv("LLM_MODEL", "claude-haiku-4-5-20251001")
llm = ChatAnthropic(
    model=llm_model,
    temperature=0.1,
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
)
print(f"LLM 초기화 완료: {llm_model}")

LLM 초기화 완료: claude-haiku-4-5-20251001


In [6]:
# State 정의 (Agentic 확장)
class RAGState(TypedDict):
    query: str
    query_plan: dict                     # classify_query() 결과
    retrieved_docs: list[Document]       # Hybrid 검색 결과 (최대 30)
    context: list[Document]              # Rerank + authority boost 후 문서
    parent_groups: list[dict]            # Parent-grouped context (expand_parents 결과)
    answer: str
    # Agentic 추가 필드
    messages: Annotated[list, add_messages]  # LLM ↔ Tool 대화 히스토리 (자동 누적)
    iteration_count: int                     # 루프 횟수 카운터 (무한루프 방지)
    fetched_chunk_ids: set[str]              # 이미 fetch한 chunk_id 집합 (중복 방지)

In [7]:
# Tool 정의: search.tools 모듈에서 import
# - fetch_paragraphs: 특정 문단 직접 조회 (교차참조 해소)
# - find_referencing_chunks: 역방향 검색 (해당 기준서를 참조하는 청크)
# - explore_related_standards: 기준서 참조 그래프 탐색

MAX_ITER = int(os.getenv("MAX_ITER", "3"))
print(f"Tool {len(TOOL_SCHEMAS)}개 로드 완료: {[t['name'] for t in TOOL_SCHEMAS]}")
print(f"MAX_ITER={MAX_ITER}")

Tool 4개 로드 완료: ['fetch_paragraphs', 'find_referencing_chunks', 'explore_related_standards', 'fetch_term_definitions']
MAX_ITER=3


In [8]:
# tool_executor 노드: search.tools.dispatch_tool 사용

def tool_executor(state: RAGState) -> dict:
    """LLM의 tool_calls를 실행: dispatch_tool로 위임."""
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage) or not last_msg.tool_calls:
        return {}

    all_new_docs = []
    tool_messages = []
    fetched = set(state.get("fetched_chunk_ids") or set())

    for tc in last_msg.tool_calls:
        new_docs, result_text = dispatch_tool(
            tool_name=tc["name"],
            args=tc["args"],
            fetched_ids=fetched,
        )
        all_new_docs.extend(new_docs)
        tool_messages.append(ToolMessage(
            content=result_text,
            tool_call_id=tc["id"],
        ))

    updated_context = (state.get("context") or []) + all_new_docs
    if len(updated_context) > 20:
        updated_context = updated_context[-20:]

    print(f"  [tool_executor] fetch {len(all_new_docs)}건, context 총 {len(updated_context)}건")
    return {
        "context": updated_context,
        "messages": tool_messages,
        "iteration_count": (state.get("iteration_count") or 0) + 1,
        "fetched_chunk_ids": fetched,
    }

print("tool_executor 정의 완료 (dispatch_tool 기반)")

tool_executor 정의 완료 (dispatch_tool 기반)


In [9]:
# 노드 1: route + retrieve (classify_query → filtered hybrid retrieval)
def retrieve(state: RAGState) -> dict:
    query = state["query"]
    plan = classify_query(query)
    print(f"  [retrieve] type={plan.query_type.value}, filter={plan.query_filter}, boost={plan.authority_boost}")

    # Dense retriever에 query_filter 적용
    dense_retriever.query_filter = plan.query_filter
    results = hybrid_retriever.invoke(query)
    retrieved = [d for d in results if d.page_content.strip()][:RETRIEVAL_K]

    return {
        "retrieved_docs": retrieved,
        "query_plan": {
            "query_type": plan.query_type.value,
            "authority_boost": plan.authority_boost,
            "query_filter": plan.query_filter,
        },
    }

In [10]:
# 노드 2: rerank + authority boost
def rerank(state: RAGState) -> dict:
    query = state["query"]
    docs = state["retrieved_docs"]
    reranked = reranker.rerank(query, docs, top_n=RERANK_TOP_N)

    # authority boost 적용 (normative 쿼리 등에서 bc/ie 점수 감쇠)
    plan = state.get("query_plan") or {}
    if plan.get("authority_boost"):
        reranked = apply_authority_boost(reranked, boost_factor=0.85)
        print(f"  [rerank] reranked={len(reranked)}건 (authority boost 적용)")
    else:
        print(f"  [rerank] reranked={len(reranked)}건")

    return {"context": reranked}

In [11]:
# 노드 3: expand_parents (reranked children → parent 그룹핑 + sibling 확장)
def expand_parents(state: RAGState) -> dict:
    docs = state["context"]
    groups = expand_to_parents(docs, max_context_chars=MAX_CONTEXT_CHARS, max_parents=MAX_PARENTS)
    total_chars = sum(sum(len(c["content"]) for c in g["children"]) for g in groups)
    print(f"  [expand_parents] {len(groups)}개 parent 그룹, 총 {total_chars:,}자")
    for g in groups:
        print(f"    - {g['standard_id']} | {g['section_type']} | {g['heading']} ({len(g['children'])}건, score={g['best_score']:.4f})")
    return {"parent_groups": groups}

print("expand_parents 노드 정의 완료")

expand_parents 노드 정의 완료


In [12]:
# context 포맷팅 함수
def format_context(docs: list[Document]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        m = doc.metadata
        std_id = m.get("standard_id", "?")
        section = m.get("section_type", "?")
        para = m.get("para_number", "?")
        score = m.get("rerank_score")

        section_label = {
            "main": "본문", "ag": "적용지침",
            "bc": "결론도출근거", "ie": "사례",
        }.get(section, section)

        header = f"[{i}] {std_id} | {section_label} | 문단 {para}"
        if score is not None:
            header += f" | relevance={score:.4f}"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

In [13]:
# K-IFRS 특화 시스템 프롬프트 (4-Tool Agentic)
SYSTEM_PROMPT = """당신은 K-IFRS(한국채택국제회계기준) 전문 회계 자문 AI입니다.

아래 규칙을 반드시 준수하세요:
1. 반드시 제공된 기준서 내용만을 근거로 답변하세요.
2. 답변 시 해당 문단 번호를 인용하세요. (예: "문단 62에 따르면...")
3. 제공된 내용으로 답변할 수 없는 경우, 솔직하게 "제공된 기준서에 해당 내용이 없습니다"라고 답변하세요.
4. 답변은 명확하고 구조적으로 작성하세요.
5. ★ 표시가 있는 문단은 검색에서 직접 매칭된 핵심 문단입니다. 주변 문단도 함께 참고하세요.

Tool 사용 지침:
- fetch_paragraphs: 컨텍스트에 "(문단 XX 참조)", "문단 XX에 따르면" 등의 교차참조가 있고,
  해당 내용이 답변의 정확성에 필요할 때 호출하세요.
- find_referencing_chunks: 특정 기준서를 다른 기준서가 어떻게 참조하는지 알아야 할 때 호출하세요.
  기준서 번호는 'K-IFRS' 접두사 없이 숫자만 사용합니다 (예: '1109').
- explore_related_standards: 주제와 관련된 다른 기준서를 폭넓게 탐색해야 할 때 호출하세요.
  기준서 번호는 숫자만 사용합니다 (예: '1115').
- fetch_term_definitions: 컨텍스트에 등장하는 회계 용어의 정확한 정의가 필요할 때 호출하세요.
  기준서 번호는 'K-IFRS' 접두사 없이 숫자만 사용합니다 (예: '1109').
  이미 용어정의가 컨텍스트에 포함되어 있으면 호출하지 마세요.
- 제공된 컨텍스트만으로 충분히 답변 가능하면 tool을 호출하지 마세요."""


# 노드 4: generate (Agentic — parent-grouped context + 4개 tool)
def generate(state: RAGState) -> dict:
    query = state["query"]
    # parent-grouped context 사용
    parent_groups = state.get("parent_groups") or []
    context_text = format_parent_context(parent_groups)
    existing_messages = list(state.get("messages") or [])

    if not existing_messages:
        new_msgs = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"다음은 K-IFRS 기준서에서 검색된 관련 내용입니다.\n"
                f"각 섹션은 같은 주제(Parent)로 묶인 문단들이며, ★ 표시는 검색 매칭된 핵심 문단입니다.\n\n"
                f"<context>\n{context_text}\n</context>\n\n"
                f"질문: {query}"),
        ]
    else:
        new_msgs = [
            HumanMessage(content=f"추가 조회된 내용을 반영하여 답변해주세요.\n\n"
                f"<updated_context>\n{context_text}\n</updated_context>"),
        ]

    llm_with_tools = llm.bind_tools(TOOL_SCHEMAS)
    response = llm_with_tools.invoke(existing_messages + new_msgs)

    iteration = state.get("iteration_count") or 0
    tool_names = [tc["name"] for tc in response.tool_calls] if response.tool_calls else []
    print(f"  [generate] iteration={iteration}, parents={len(parent_groups)}개, context={len(context_text):,}자, tool_calls={tool_names or 'none'}")

    if response.tool_calls:
        return {"messages": new_msgs + [response], "answer": ""}
    else:
        if isinstance(response.content, str):
            answer = response.content
        else:
            answer = next(
                (b["text"] for b in response.content if isinstance(b, dict) and b.get("type") == "text"),
                str(response.content),
            )
        return {"messages": new_msgs + [response], "answer": answer}

In [14]:
# should_continue: tool 호출 여부 판단
def should_continue(state: RAGState) -> str:
    """generate 후 분기: tool_calls 있으면 tool_executor, 없으면 종료."""
    if (state.get("iteration_count") or 0) >= MAX_ITER:
        print(f"  [should_continue] MAX_ITER({MAX_ITER}) 도달 → 종료")
        return "end"
    messages = state.get("messages") or []
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
            print(f"  [should_continue] tool_calls 감지 → tool_executor")
            return "tool_executor"
    return "end"


# LangGraph 구성 (Agentic + Parent-Grouped)
graph_builder = StateGraph(RAGState)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("rerank", rerank)
graph_builder.add_node("expand_parents", expand_parents)
graph_builder.add_node("generate", generate)
graph_builder.add_node("tool_executor", tool_executor)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "rerank")
graph_builder.add_edge("rerank", "expand_parents")
graph_builder.add_edge("expand_parents", "generate")
graph_builder.add_conditional_edges("generate", should_continue, {
    "tool_executor": "tool_executor",
    "end": END,
})
graph_builder.add_edge("tool_executor", "generate")

rag_chain = graph_builder.compile()
print("Agentic RAG Chain 구성 완료: START → retrieve → rerank → expand_parents → generate ⇄ tool_executor → END")

Agentic RAG Chain 구성 완료: START → retrieve → rerank → expand_parents → generate ⇄ tool_executor → END


In [15]:
# 그래프 시각화
from IPython.display import Image, display

try:
    display(Image(rag_chain.get_graph().draw_mermaid_png()))
except Exception:
    print(rag_chain.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	rerank(rerank)
	expand_parents(expand_parents)
	generate(generate)
	tool_executor(tool_executor)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	expand_parents --> generate;
	generate -. &nbsp;end&nbsp; .-> __end__;
	generate -.-> tool_executor;
	rerank --> expand_parents;
	retrieve --> rerank;
	tool_executor --> generate;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. 테스트

상호참조가 많은 쿼리로 tool 호출 루프를 검증

In [16]:
QUERIES = [
    "수익인식의 5단계에 대해서 상세히 설명해주세요.",
    "금융자산의 기대신용손실은 어떻게 측정하나요?",
    "수행의무를 식별하고 거래가격을 배분하는 기준은?",
    "연결재무제표 작성 시 지배력 판단 기준은?",
    "유형자산의 감가상각 방법에는 어떤 것들이 있나요?",
]

In [17]:
from IPython.display import Markdown

In [20]:
# Agentic RAG 테스트 (상호참조 자동 검색 포함)
result = rag_chain.invoke({
    "query": QUERIES[2],
    "messages": [],
    "iteration_count": 0,
    "fetched_chunk_ids": set(),
})

# 결과 요약
print(f"iteration_count: {result.get('iteration_count', 0)}")
print(f"context 문서 수: {len(result.get('context', []))}")
print(f"messages 수: {len(result.get('messages', []))}")
print(f"fetched_chunk_ids: {result.get('fetched_chunk_ids', set())}")
print()

# tool로 추가된 문서 표시
tool_fetched = [d for d in result.get("context", []) if d.metadata.get("fetched_by_tool")]
if tool_fetched:
    print(f"--- Tool로 추가 fetch된 문서 ({len(tool_fetched)}건) ---")
    for d in tool_fetched:
        m = d.metadata
        print(f"  {m.get('standard_id')} | {m.get('section_type')} | 문단 {m.get('para_number')}")
else:
    print("--- Tool 호출 없이 직접 답변 ---")

print()
display(Markdown(result["answer"]))

  [retrieve] type=normative, filter={'section_type__in': ['main', 'ag']}, boost=True
  [rerank] reranked=15건 (authority boost 적용)
  [expand_parents] 1개 parent 그룹, 총 10,788자
    - K-IFRS 1115 | main | 측정 (43건, score=0.9998)
  [generate] iteration=0, parents=1개, context=11,273자, tool_calls=none
iteration_count: 0
context 문서 수: 15
messages 수: 3
fetched_chunk_ids: set()

--- Tool 호출 없이 직접 답변 ---



# 수행의무 식별 및 거래가격 배분 기준

제공하신 컨텍스트에서 거래가격 배분에 관한 내용을 확인할 수 있습니다. 수행의무 식별에 대한 구체적인 기준은 제시된 문단에 직접 나타나지 않으나, 거래가격 배분의 기준은 명확히 규정되어 있습니다.

## 거래가격 배분의 기본 원칙

**문단 74**에 따르면, 거래가격은 **상대적 개별 판매가격을 기준으로** 계약에서 식별된 각 수행의무에 배분합니다. 다만 할인액 배분(문단 81~83)과 변동금액 배분(문단 84~86)의 특수한 경우는 제외됩니다.

## 개별 판매가격에 기초한 배분

**문단 76**에서는 다음과 같이 규정합니다:
- 계약 개시시점에 계약상 각 수행의무의 대상인 구별되는 재화나 용역의 **개별 판매가격을 산정**
- 이 개별 판매가격에 **비례하여 거래가격을 배분**

**문단 77**에 따르면, 개별 판매가격은:
- 기업이 고객에게 약속한 재화나 용역을 **별도로 판매할 경우의 가격**
- 최선의 증거는 **비슷한 상황에서 비슷한 고객에게 별도로 판매할 때의 관측 가능한 가격**

## 개별 판매가격 추정 방법

**문단 79**에서는 다음 세 가지 방법을 제시합니다:

1. **시장평가 조정 접근법** - 시장 평가를 통해 고객이 지급하려는 가격을 추정하고, 경쟁자 가격을 참조하여 조정

2. **예상원가 이윤 가산 접근법** - 수행의무 이행을 위한 예상원가에 적절한 이윤을 가산

3. **잔여접근법** - 총 거래가격에서 다른 재화나 용역의 관측 가능한 개별 판매가격 합계를 차감 (문단 79⑶의 특정 기준 충족 시에만 사용)

## 거래가격 변동 시 배분

**문단 88**에 따르면:
- 거래가격의 후속 변동은 **계약 개시시점과 같은 기준으로** 수행의무에 배분
- 계약 개시 후 개별 판매가격 변동을 반영하기 위해 거래가격을 **다시 배분하지 않음**

이는 K-IFRS 1115의 핵심 원칙으로, 거래가격 배분의 일관성과 예측가능성을 보장합니다.

In [ ]:
result

In [ ]:
close_pool()
print("PostgreSQL 커넥션 풀 종료")